In [ ]:
%pip install -q pandas numpy scikit-learn scipy joblib matplotlib

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import joblib
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
    joblib = None

from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC

print('Libraries loaded successfully.')
print('joblib available:', HAS_JOBLIB)

In [ ]:
TRAIN_PATH = 'train.csv'
VALID_PATH = 'valid.csv'
TEST_PATH = 'test.csv'
OUT_DIR = 'food_outputs_notebook'

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('Output directory:', Path(OUT_DIR).resolve())

In [ ]:
# =========================
# Metric and preprocessing
# =========================

def official_st1_score(
    y_hazard_true: Union[pd.Series, np.ndarray, List[str]],
    y_hazard_pred: Union[pd.Series, np.ndarray, List[str]],
    y_product_true: Union[pd.Series, np.ndarray, List[str]],
    y_product_pred: Union[pd.Series, np.ndarray, List[str]],
) -> Tuple[float, float, float, int]:
    y_hazard_true = np.asarray(y_hazard_true)
    y_hazard_pred = np.asarray(y_hazard_pred)
    y_product_true = np.asarray(y_product_true)
    y_product_pred = np.asarray(y_product_pred)

    f1_hazard = f1_score(y_hazard_true, y_hazard_pred, average='macro')
    correct_hazard_mask = y_hazard_true == y_hazard_pred

    if int(correct_hazard_mask.sum()) == 0:
        f1_product_when_hazard_correct = 0.0
    else:
        f1_product_when_hazard_correct = f1_score(
            y_product_true[correct_hazard_mask],
            y_product_pred[correct_hazard_mask],
            average='macro',
        )

    score = (f1_hazard + f1_product_when_hazard_correct) / 2.0
    return score, f1_hazard, f1_product_when_hazard_correct, int(correct_hazard_mask.sum())


def load_df(path: Union[str, Path]) -> pd.DataFrame:
    df = pd.read_csv(path).copy()
    for col in ['title', 'text', 'country']:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str)
    for col in ['year', 'month', 'day']:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int).astype(str)
    return df


def make_text(df: pd.DataFrame, mode: str) -> pd.Series:
    title = df.get('title', pd.Series('', index=df.index)).astype(str)
    text = df.get('text', pd.Series('', index=df.index)).astype(str)
    country = df.get('country', pd.Series('', index=df.index)).astype(str)
    year = df.get('year', pd.Series('0', index=df.index)).astype(str)
    month = df.get('month', pd.Series('0', index=df.index)).astype(str)
    day = df.get('day', pd.Series('0', index=df.index)).astype(str)
    date = year + ' ' + month + ' ' + day

    if mode == 'title':
        return title
    if mode == 'text':
        return text
    if mode == 'title_text':
        return title + ' [SEP] ' + text
    if mode == 'title_text700':
        return title + ' [SEP] ' + text.str.slice(0, 700)
    if mode == 'rich700':
        return (
            'title: ' + title +
            ' country: ' + country +
            ' date: ' + date +
            ' text: ' + text.str.slice(0, 700)
        )
    raise ValueError(f'Unknown mode: {mode}')

In [ ]:
# =========================
# Model wrappers
# =========================

class FoodModelBase:
    def fit(self, train_df: pd.DataFrame) -> 'FoodModelBase':
        raise NotImplementedError

    def predict(self, df: pd.DataFrame):
        raise NotImplementedError


class MajorityBaseline(FoodModelBase):
    def __init__(self):
        self.major_hazard = None
        self.major_product = None

    def fit(self, train_df: pd.DataFrame):
        self.major_hazard = train_df['hazard-category'].value_counts().idxmax()
        self.major_product = train_df['product-category'].value_counts().idxmax()
        return self

    def predict(self, df: pd.DataFrame):
        hazard_pred = np.full(len(df), self.major_hazard, dtype=object)
        product_pred = np.full(len(df), self.major_product, dtype=object)
        return hazard_pred, product_pred


class DualPipelineModel(FoodModelBase):
    def __init__(self, hazard_pipeline, product_pipeline, hazard_text_mode: str, product_text_mode: str):
        self.hazard_pipeline = hazard_pipeline
        self.product_pipeline = product_pipeline
        self.hazard_text_mode = hazard_text_mode
        self.product_text_mode = product_text_mode

    def fit(self, train_df: pd.DataFrame):
        self.hazard_pipeline.fit(make_text(train_df, self.hazard_text_mode), train_df['hazard-category'])
        self.product_pipeline.fit(make_text(train_df, self.product_text_mode), train_df['product-category'])
        return self

    def predict(self, df: pd.DataFrame):
        hazard_pred = self.hazard_pipeline.predict(make_text(df, self.hazard_text_mode))
        product_pred = self.product_pipeline.predict(make_text(df, self.product_text_mode))
        return hazard_pred, product_pred


class HazardAwareProductModel(FoodModelBase):
    def __init__(self, hazard_pipeline, product_pipeline_factory, hazard_text_mode: str, product_text_mode: str):
        self.hazard_pipeline = hazard_pipeline
        self.product_pipeline_factory = product_pipeline_factory
        self.hazard_text_mode = hazard_text_mode
        self.product_text_mode = product_text_mode
        self.product_models_by_hazard: Dict[str, Union[str, Pipeline]] = {}

    def fit(self, train_df: pd.DataFrame):
        self.hazard_pipeline.fit(make_text(train_df, self.hazard_text_mode), train_df['hazard-category'])

        for hazard_label, sub_df in train_df.groupby('hazard-category'):
            unique_products = sorted(sub_df['product-category'].unique())
            if len(unique_products) == 1:
                self.product_models_by_hazard[hazard_label] = unique_products[0]
            else:
                product_pipe = self.product_pipeline_factory()
                product_pipe.fit(make_text(sub_df, self.product_text_mode), sub_df['product-category'])
                self.product_models_by_hazard[hazard_label] = product_pipe
        return self

    def predict(self, df: pd.DataFrame):
        hazard_pred = self.hazard_pipeline.predict(make_text(df, self.hazard_text_mode))
        product_text = make_text(df, self.product_text_mode)
        product_pred = []

        for i, hazard_label in enumerate(hazard_pred):
            model_or_label = self.product_models_by_hazard[hazard_label]
            if isinstance(model_or_label, str):
                product_pred.append(model_or_label)
            else:
                product_pred.append(model_or_label.predict([product_text.iloc[i]])[0])

        return np.asarray(hazard_pred), np.asarray(product_pred)

In [ ]:
# =========================
# Pipeline builders
# =========================

def word_svc_pipeline(c: float = 1.0, ngram_range=(1, 2), min_df: int = 2, max_df: float = 0.98):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=True,
        )),
        ('clf', LinearSVC(C=c, class_weight='balanced')),
    ])


def char_svc_pipeline(c: float = 1.0, ngram_range=(3, 5), min_df: int = 2):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=ngram_range,
            min_df=min_df,
            sublinear_tf=True,
        )),
        ('clf', LinearSVC(C=c, class_weight='balanced')),
    ])


def union_svc_pipeline(c: float, word_ngram_range, char_ngram_range, word_min_df: int, word_max_df: float, char_min_df: int):
    return Pipeline([
        ('features', FeatureUnion([
            ('word', TfidfVectorizer(
                lowercase=True,
                strip_accents='unicode',
                ngram_range=word_ngram_range,
                min_df=word_min_df,
                max_df=word_max_df,
                sublinear_tf=True,
            )),
            ('char', TfidfVectorizer(
                analyzer='char_wb',
                ngram_range=char_ngram_range,
                min_df=char_min_df,
                sublinear_tf=True,
            )),
        ])),
        ('clf', LinearSVC(C=c, class_weight='balanced')),
    ])


def sgd_word_pipeline(loss: str = 'modified_huber', alpha: float = 2e-5):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
        )),
        ('clf', SGDClassifier(
            loss=loss,
            alpha=alpha,
            penalty='l2',
            class_weight='balanced',
            max_iter=4000,
            random_state=42,
        )),
    ])


def complement_nb_pipeline(alpha: float = 0.5):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            norm=None,
        )),
        ('clf', ComplementNB(alpha=alpha)),
    ])

In [ ]:
# =========================
# Method zoo
# =========================

def build_method(method_name: str):
    if method_name == 'majority_baseline':
        return MajorityBaseline()

    if method_name == 'complement_nb_word':
        return DualPipelineModel(
            hazard_pipeline=complement_nb_pipeline(alpha=0.5),
            product_pipeline=complement_nb_pipeline(alpha=0.5),
            hazard_text_mode='rich700',
            product_text_mode='rich700',
        )

    if method_name == 'sgd_word':
        return DualPipelineModel(
            hazard_pipeline=sgd_word_pipeline(loss='modified_huber', alpha=2e-5),
            product_pipeline=sgd_word_pipeline(loss='modified_huber', alpha=2e-5),
            hazard_text_mode='rich700',
            product_text_mode='rich700',
        )

    if method_name == 'svc_word_both':
        return DualPipelineModel(
            hazard_pipeline=word_svc_pipeline(c=1.0, ngram_range=(1, 2), min_df=2, max_df=0.98),
            product_pipeline=word_svc_pipeline(c=1.0, ngram_range=(1, 2), min_df=2, max_df=0.98),
            hazard_text_mode='rich700',
            product_text_mode='rich700',
        )

    if method_name == 'svc_specialized':
        return DualPipelineModel(
            hazard_pipeline=word_svc_pipeline(c=1.0, ngram_range=(1, 2), min_df=2, max_df=0.98),
            product_pipeline=char_svc_pipeline(c=1.0, ngram_range=(3, 5), min_df=2),
            hazard_text_mode='title_text700',
            product_text_mode='title',
        )

    if method_name == 'svc_hazard_aware':
        return HazardAwareProductModel(
            hazard_pipeline=word_svc_pipeline(c=1.0, ngram_range=(1, 2), min_df=2, max_df=0.98),
            product_pipeline_factory=lambda: char_svc_pipeline(c=1.0, ngram_range=(3, 5), min_df=1),
            hazard_text_mode='title_text700',
            product_text_mode='title',
        )

    if method_name == 'svc_union_hazard_char_product':
        return DualPipelineModel(
            hazard_pipeline=union_svc_pipeline(
                c=0.8,
                word_ngram_range=(1, 2),
                char_ngram_range=(3, 5),
                word_min_df=2,
                word_max_df=0.98,
                char_min_df=2,
            ),
            product_pipeline=char_svc_pipeline(c=1.0, ngram_range=(3, 5), min_df=2),
            hazard_text_mode='title_text700',
            product_text_mode='title',
        )

    if method_name == 'svc_union_best':
        return DualPipelineModel(
            hazard_pipeline=union_svc_pipeline(
                c=0.8,
                word_ngram_range=(1, 2),
                char_ngram_range=(3, 5),
                word_min_df=2,
                word_max_df=0.98,
                char_min_df=2,
            ),
            product_pipeline=union_svc_pipeline(
                c=1.0,
                word_ngram_range=(1, 2),
                char_ngram_range=(3, 5),
                word_min_df=1,
                word_max_df=1.0,
                char_min_df=2,
            ),
            hazard_text_mode='title_text700',
            product_text_mode='title',
        )

    raise ValueError(f'Unknown method: {method_name}')


DEFAULT_METHODS = [
    'majority_baseline',
    'complement_nb_word',
    'sgd_word',
    'svc_word_both',
    'svc_specialized',
    'svc_hazard_aware',
    'svc_union_hazard_char_product',
    'svc_union_best',
]

print('Methods available:', DEFAULT_METHODS)

In [ ]:
# =========================
# Load data
# =========================

train_df = load_df(TRAIN_PATH)
valid_df = load_df(VALID_PATH)
test_df = load_df(TEST_PATH)

print('train shape:', train_df.shape)
print('valid shape:', valid_df.shape)
print('test shape:', test_df.shape)

train_df.head(3)

In [ ]:
# =========================
# Evaluate all methods on valid
# =========================

results = []

for method_name in DEFAULT_METHODS:
    print(f'=== Evaluating: {method_name} ===')
    model = build_method(method_name)
    model.fit(train_df)
    hazard_pred, product_pred = model.predict(valid_df)

    score, f1_hazard, f1_product_cond, correct_hazard = official_st1_score(
        valid_df['hazard-category'],
        hazard_pred,
        valid_df['product-category'],
        product_pred,
    )

    results.append({
        'method': method_name,
        'official_score': score,
        'f1_hazard_macro': f1_hazard,
        'f1_product_when_hazard_correct_macro': f1_product_cond,
        'hazard_correct_count': correct_hazard,
    })

    pred_df = pd.DataFrame({
        'id': valid_df.get('Unnamed: 0', pd.Series(np.arange(len(valid_df)))).values,
        'hazard-category': hazard_pred,
        'product-category': product_pred,
    })
    pred_df.to_csv(Path(OUT_DIR) / f'valid_predictions__{method_name}.csv', index=False)

results_df = pd.DataFrame(results).sort_values('official_score', ascending=False).reset_index(drop=True)
results_df.to_csv(Path(OUT_DIR) / 'method_scores.csv', index=False)
Path(OUT_DIR, 'method_scores.json').write_text(results_df.to_json(orient='records', indent=2), encoding='utf-8')

print('Saved scores to:', Path(OUT_DIR) / 'method_scores.csv')
results_df

In [ ]:
# Plot των scores 

plt.figure(figsize=(10, 5))
plt.bar(results_df['method'], results_df['official_score'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Official ST1 score')
plt.title('Comparison of methods on valid.csv')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Train best method on train+valid and create Kaggle submission
# =========================

BEST_METHOD = results_df.iloc[0]['method']
print('Best method based on valid:', BEST_METHOD)

train_full = pd.concat([train_df, valid_df], axis=0, ignore_index=True)
best_model = build_method(BEST_METHOD)
best_model.fit(train_full)

hazard_test_pred, product_test_pred = best_model.predict(test_df)

submission_df = pd.DataFrame({
    'id': test_df['Unnamed: 0'] if 'Unnamed: 0' in test_df.columns else np.arange(len(test_df)),
    'hazard-category': hazard_test_pred,
    'product-category': product_test_pred,
})

submission_path = Path(OUT_DIR) / f'submission__{BEST_METHOD}.csv'
submission_df.to_csv(submission_path, index=False)
print('Saved submission to:', submission_path)

submission_df.head()

In [ ]:
# save model

if HAS_JOBLIB:
    model_path = Path(OUT_DIR) / f'model__{BEST_METHOD}.joblib'
    joblib.dump(best_model, model_path)
    print('Saved model to:', model_path)
else:
    print('joblib not installed, so model was not saved. The submission CSV was still created normally.')